<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="400" height="600" /> 
    
    
## <font color='navy'> Tarea 5. Regresión Lineal: Ingreso mensual en México (ENOE 2025-T1)
    
### <font color='navy'> Análisis Estadístico Multivariado

### <font color='navy'> Parte 1. Unión de las tablas COE1 y COE2

La **Encuesta Nacional de Ocupación y Empleo (ENOE)** del INEGI aplica el *Cuestionario de Ocupación y Empleo* (versión ampliada) a la población de 12 años y más. Por su tamaño, el cuestionario se publica dividido en dos tablas:

| **Tabla** | **Contenido** |
|-----------|---------------|
| **COE1** | Preguntas 1 a 5: condición de actividad, características del empleo (sindicato, contrato, prestaciones, tamaño de empresa, antigüedad) y jornada de trabajo. |
| **COE2** | Preguntas 6 a 11: ingresos (periodicidad y monto), segundo trabajo, búsqueda de empleo y uso del tiempo (capacitación, etc.). |

Ambas tablas tienen **un registro por persona** y comparten los mismos campos de identificación, por lo que se pueden unir para continuar la secuencia del cuestionario.

Como nuestras variables de interés están repartidas en ambas tablas (`p3i`, `p3j`, `p3m*`, `p3q`, `p3r_anio`, `p5e_*` en COE1; `p6b1`, `p6b2`, `p7g*`, `p11_h1` en COE2), en este notebook las unimos en un solo dataset.

In [1]:
# Cargar librerias
import pandas as pd
import numpy as np

#### Paso 1. Cargar los datos

Los CSV del INEGI guardan las respuestas vacías como un espacio en blanco (`' '`), así que las leemos como `NaN` con `na_values`.

In [2]:
# Rutas de los archivos
ruta = 'conjunto_de_datos_enoe_2025_1t_csv/'
ruta_coe1 = ruta + 'conjunto_de_datos_coe1_enoe_2025_1t/conjunto_de_datos/conjunto_de_datos_coe1_enoe_2025_1t.csv'
ruta_coe2 = ruta + 'conjunto_de_datos_coe2_enoe_2025_1t/conjunto_de_datos/conjunto_de_datos_coe2_enoe_2025_1t.csv'

In [3]:
# Cargar datos
coe1 = pd.read_csv(ruta_coe1, na_values=' ', low_memory=False)
coe2 = pd.read_csv(ruta_coe2, na_values=' ', low_memory=False)

print('COE1:', coe1.shape)
print('COE2:', coe2.shape)

COE1: (345626, 189)
COE2: (345626, 137)


In [4]:
coe1.head()

,r_def,cd_a,ent,con,upm,d_sem,n_pro_viv,v_sel,n_hog,h_mud,...,p5g13,p5g14,p5g15,p5g99,p5h,ur,tipo,mes_cal,fac_tri,fac_men
0,0,1,9,40002,905953,305,4,51,1,0,...,NaN,14.0,NaN,NaN,NaN,1,1,1,1286,4109
1,0,1,9,40002,905953,305,4,51,1,0,...,NaN,NaN,NaN,NaN,NaN,1,1,1,1286,4109
2,0,1,9,40009,917468,101,199,3,1,0,...,NaN,NaN,NaN,NaN,NaN,1,1,96,1518,0
3,0,1,9,40009,917468,101,199,3,1,0,...,NaN,14.0,NaN,NaN,NaN,1,1,96,1518,0
4,0,1,9,40014,911027,302,99,54,1,1,...,NaN,14.0,NaN,NaN,NaN,1,1,1,1247,3983


In [5]:
coe2.head()

,cd_a,ent,con,upm,d_sem,n_pro_viv,v_sel,n_hog,h_mud,n_ent,...,p11_h7,p11_m7,p11_8,p11_h8,p11_m8,ur,tipo,mes_cal,fac_tri,fac_men
0,1,9,40003,906619,101,77,4,1,0,5,...,35.0,0.0,NaN,NaN,NaN,1,1,96,464,0
1,1,9,40003,906619,101,77,4,1,0,5,...,35.0,0.0,NaN,NaN,NaN,1,1,96,464,0
2,1,9,40003,906619,101,77,4,1,0,5,...,NaN,NaN,NaN,NaN,NaN,1,1,96,464,0
3,1,9,40005,906533,310,168,53,1,0,3,...,10.0,0.0,NaN,NaN,NaN,1,1,3,1019,3336
4,1,9,40006,906681,309,42,52,1,0,3,...,35.0,0.0,NaN,NaN,NaN,1,1,2,798,2467


#### Paso 2. Definir la llave de unión

De acuerdo con el documento del INEGI *Conociendo la base de datos de la ENOE*, la llave para relacionar COE1 y COE2 está formada por:

| **Campo** | **Descripción** |
|-----------|-----------------|
| `cd_a` | Ciudad autorrepresentada |
| `ent` | Entidad |
| `con` | Control |
| `v_sel` | Vivienda seleccionada |
| `n_hog` | Número de hogar |
| `h_mud` | Hogar mudado |
| `n_ren` | Número de renglón (persona dentro del hogar) |

Sin embargo, ese documento es de la época en la que todas las entrevistas eran cara a cara. Desde la ENOE nueva se combinan entrevistas **cara a cara y telefónicas** (`tipo`) y se registra el **mes calendario** (`mes_cal`), por lo que la llave de 7 campos ya no identifica a una persona de forma única. Lo comprobamos:

In [6]:
# Llave documentada (ENOE 2010)
llave_2010 = ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

print('Duplicados en COE1:', coe1.duplicated(llave_2010).sum())
print('Duplicados en COE2:', coe2.duplicated(llave_2010).sum())

Duplicados en COE1: 2674
Duplicados en COE2: 2674


Si uniéramos con esta llave tendríamos una unión *muchos a muchos*: el número de registros crecería y se mezclarían personas distintas (por ejemplo, con edades diferentes en cada tabla). Agregando `tipo` y `mes_cal` la llave se vuelve única:

In [7]:
# Llave completa
llave = ['tipo', 'mes_cal', 'cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

print('Duplicados en COE1:', coe1.duplicated(llave).sum())
print('Duplicados en COE2:', coe2.duplicated(llave).sum())

Duplicados en COE1: 0
Duplicados en COE2: 0


#### Paso 3. Unir las tablas

Además de la llave, ambas tablas repiten algunas columnas de identificación (edad, UPM, ponderadores, etc.). Antes de unir verificamos que, para la misma persona, esas columnas coincidan. Si coinciden, solo conservamos la copia de COE1 para no duplicar columnas.

In [8]:
# Columnas que aparecen en ambas tablas pero no forman parte de la llave
comunes = [c for c in coe1.columns if c in coe2.columns and c not in llave]
comunes

['upm',
 'd_sem',
 'n_pro_viv',
 'n_ent',
 'per',
 'eda',
 'n_inf',
 'ur',
 'fac_tri',
 'fac_men']

In [9]:
# Unión de prueba con indicador para validar la correspondencia 1 a 1
prueba = coe1[llave + comunes].merge(coe2[llave + comunes], on=llave, how='outer',
                                     suffixes=('_coe1', '_coe2'), indicator=True)
print(prueba['_merge'].value_counts())

# Diferencias en las columnas comunes
diferencias = {c: (prueba[c + '_coe1'].fillna(-1) != prueba[c + '_coe2'].fillna(-1)).sum() for c in comunes}
pd.Series(diferencias, name='registros distintos')

_merge
both          345626
left_only          0
right_only         0
Name: count, dtype: int64


upm          0
d_sem        0
n_pro_viv    0
n_ent        0
per          0
eda          0
n_inf        0
ur           0
fac_tri      0
fac_men      0
Name: registros distintos, dtype: int64

Todos los registros están en ambas tablas (`both`) y las columnas comunes coinciden, así que hacemos la unión definitiva. Usamos `validate='one_to_one'` para que pandas marque error si la llave no fuera única.

In [10]:
# Unir COE1 con las columnas propias de COE2
coe = coe1.merge(coe2.drop(columns=comunes), on=llave, how='inner', validate='one_to_one')

print('COE1:', coe1.shape)
print('COE2:', coe2.shape)
print('COE :', coe.shape)

COE1: (345626, 189)
COE2: (345626, 137)
COE : (345626, 307)


In [11]:
coe.head()

,r_def,cd_a,ent,con,upm,d_sem,n_pro_viv,v_sel,n_hog,h_mud,...,p11_m5,p11_6,p11_h6,p11_m6,p11_7,p11_h7,p11_m7,p11_8,p11_h8,p11_m8
0,0,1,9,40002,905953,305,4,51,1,0,...,NaN,NaN,NaN,NaN,7.0,98.0,0.0,NaN,NaN,NaN
1,0,1,9,40002,905953,305,4,51,1,0,...,NaN,NaN,NaN,NaN,7.0,3.0,30.0,NaN,NaN,NaN
2,0,1,9,40009,917468,101,199,3,1,0,...,NaN,6.0,5.0,0.0,7.0,21.0,0.0,NaN,NaN,NaN
3,0,1,9,40009,917468,101,199,3,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,1,9,40014,911027,302,99,54,1,1,...,NaN,NaN,NaN,NaN,7.0,98.0,0.0,NaN,NaN,NaN


#### Paso 4. Verificar las variables de interés

Revisamos que las variables seleccionadas existan en el dataset unido. Algunos nombres del cuestionario cambian en la base de datos:

+ **5e** (tiempo trabajado habitualmente) viene desglosada por día (`p5e_hlu`, `p5e_mlu`, ...); el total está en `p5e_thrs` (horas) y `p5e_tdia` (días).
+ **11 opción 1** (capacitación) se guarda en `p11_1` (sí/no), `p11_h1` (horas) y `p11_m1` (minutos).
+ **7g** (forma de pago del segundo trabajo) está en `p7g1` (efectivo), `p7g2` (especie) y el monto en `p7gcan`.

In [12]:
# Variables seleccionadas
variables = {
    'eda':      'Edad',
    'p6b1':     'Periodicidad de pago',
    'p6b2':     'Monto del pago',
    'p7g1':     '2do trabajo: pago en efectivo',
    'p7g2':     '2do trabajo: pago en especie',
    'p7gcan':   '2do trabajo: monto',
    'p3i':      'Pertenece a sindicato',
    'p3j':      'Tiene contrato por escrito',
    'p3q':      'Tamaño de la empresa',
    'p3m1':     'Prestación: crédito vivienda',
    'p3m2':     'Prestación: guardería',
    'p3m3':     'Prestación: cuidados maternos/paternos',
    'p3m4':     'Prestación: fondo de retiro',
    'p3m5':     'Prestación: seguro de vida',
    'p3m6':     'Prestación: gastos médicos privados',
    'p3m7':     'Prestación: préstamos / caja de ahorro',
    'p3r_anio': 'Año de ingreso al empleo',
    'p5e_thrs': 'Horas trabajadas a la semana (habitual)',
    'p5e_tdia': 'Días trabajados a la semana (habitual)',
    'p11_h1':   'Horas de capacitación',
    'p11_m1':   'Minutos de capacitación',
}

pd.DataFrame({'descripcion': variables.values(),
              'existe': [v in coe.columns for v in variables],
              'no_nulos': [coe[v].notna().sum() if v in coe.columns else 0 for v in variables]},
             index=variables.keys())

,descripcion,existe,no_nulos
eda,Edad,True,345626
p6b1,Periodicidad de pago,True,182353
p6b2,Monto del pago,True,119697
p7g1,2do trabajo: pago en efectivo,True,6881
p7g2,2do trabajo: pago en especie,True,1
p7gcan,2do trabajo: monto,True,6882
p3i,Pertenece a sindicato,True,135571
p3j,Tiene contrato por escrito,True,135571
p3q,Tamaño de la empresa,True,141701
p3m1,Prestación: crédito vivienda,True,68672


#### Paso 5. Guardar el dataset unido

In [13]:
# Guardar en CSV
coe.to_csv('enoe_2025_1t_coe.csv', index=False)